# 📊 Customer Churn Prediction
**Dataset:** IBM Telco Customer Churn (Kaggle)  
**Link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
**Goal:** Predict which customers are likely to churn using ML models

---
## Table of Contents
1. [Setup & Data Loading](#1)
2. [EDA - Exploratory Data Analysis](#2)
3. [Data Cleaning](#3)
4. [Feature Engineering](#4)
5. [Model Training](#5)
6. [Model Comparison & Evaluation](#6)
7. [Save Best Model](#7)

## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
import joblib
import os

# Plot styling
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────────────
# Dataset: IBM Telco Customer Churn
# Kaggle Link: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

df = pd.read_csv('../data/telco_churn.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Summary statistics
df.describe(include='all').T

## 2. EDA - Exploratory Data Analysis <a id='2'></a>

In [ ]:
# ── 2.1 Target Variable Distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

churn_counts = df['Churn'].value_counts()
colors = ['#2ecc71', '#e74c3c']

# Bar chart
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: Customer Churn', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../models/01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Churn Rate: {churn_counts["Yes"]/len(df)*100:.1f}%')

In [ ]:
# ── 2.2 Missing Values Analysis ───────────────────────────────────────────────
# TotalCharges has spaces as missing values — fix first
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df.isnull().sum()
missing = missing[missing > 0]

print('🔍 Missing Values:')
print(missing)
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# Visualize
plt.figure(figsize=(10, 5))
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if len(missing_pct) > 0:
    bars = plt.bar(missing_pct.index, missing_pct.values, color='#e74c3c', edgecolor='white')
    for bar, val in zip(bars, missing_pct.values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}%', ha='center', fontweight='bold')
    plt.title('Missing Values (%)', fontsize=14, fontweight='bold')
    plt.xlabel('Column')
    plt.ylabel('Missing %')
    plt.savefig('../models/02_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing values after conversion!')

In [ ]:
# ── 2.3 Numerical Feature Distributions ──────────────────────────────────────
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, col in enumerate(num_cols):
    # Distribution by Churn
    for churn_val, color in zip(['No', 'Yes'], ['#2ecc71', '#e74c3c']):
        subset = df[df['Churn'] == churn_val][col].dropna()
        axes[0, idx].hist(subset, bins=30, alpha=0.6, color=color, label=f'Churn={churn_val}', edgecolor='white')
    axes[0, idx].set_title(f'{col} Distribution by Churn', fontweight='bold')
    axes[0, idx].set_xlabel(col)
    axes[0, idx].set_ylabel('Frequency')
    axes[0, idx].legend()
    
    # Box plot
    data_no = df[df['Churn'] == 'No'][col].dropna()
    data_yes = df[df['Churn'] == 'Yes'][col].dropna()
    bp = axes[1, idx].boxplot([data_no, data_yes], labels=['No Churn', 'Churn'],
                               patch_artist=True,
                               boxprops=dict(facecolor='lightblue', alpha=0.7))
    bp['boxes'][1].set_facecolor('#ffcccc')
    axes[1, idx].set_title(f'{col} Boxplot by Churn', fontweight='bold')
    axes[1, idx].set_ylabel(col)

plt.suptitle('Numerical Features vs Churn', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/03_numerical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.4 Categorical Features vs Churn ────────────────────────────────────────
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'InternetService', 'Contract', 'PaymentMethod', 'PaperlessBilling']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
    churn_rate.columns = [col, 'ChurnRate']
    churn_rate = churn_rate.sort_values('ChurnRate', ascending=False)
    
    bars = axes[idx].bar(churn_rate[col].astype(str), churn_rate['ChurnRate'],
                          color=sns.color_palette('Set2', len(churn_rate)),
                          edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, churn_rate['ChurnRate']):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
    
    axes[idx].set_title(f'Churn Rate by {col}', fontweight='bold', fontsize=11)
    axes[idx].set_ylabel('Churn Rate (%)')
    axes[idx].tick_params(axis='x', rotation=15)
    axes[idx].set_ylim(0, churn_rate['ChurnRate'].max() * 1.2)

plt.suptitle('Churn Rate by Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/04_categorical_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.5 Correlation Heatmap ───────────────────────────────────────────────────
# Encode target and numeric columns for correlation
df_corr = df.copy()
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)

# Label encode categoricals for correlation only
le = LabelEncoder()
encode_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
               'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
               'PaperlessBilling', 'PaymentMethod']

for col in encode_cols:
    df_corr[col + '_enc'] = le.fit_transform(df_corr[col].astype(str))

corr_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_num'] + \
            [c + '_enc' for c in ['Contract', 'InternetService', 'TechSupport', 
                                   'OnlineSecurity', 'PaperlessBilling', 'PaymentMethod']]

corr_matrix = df_corr[corr_cols].corr()

# Rename for readability
rename_map = {c + '_enc': c for c in ['Contract', 'InternetService', 'TechSupport',
                                        'OnlineSecurity', 'PaperlessBilling', 'PaymentMethod']}
rename_map['Churn_num'] = 'Churn'
corr_matrix = corr_matrix.rename(columns=rename_map, index=rename_map)

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 9}, square=True)
plt.title('Correlation Heatmap — Feature vs Churn', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../models/05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n🔑 Top correlations with Churn:')
print(corr_matrix['Churn'].drop('Churn').abs().sort_values(ascending=False).head(8))

## 3. Data Cleaning <a id='3'></a>

In [ ]:
# ── 3.1 Clean & Prepare ───────────────────────────────────────────────────────
df_clean = df.copy()

# 1. Drop customerID (not useful)
df_clean.drop('customerID', axis=1, inplace=True)
print('✅ Dropped customerID')

# 2. Fix TotalCharges (convert to numeric, already done above)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
print(f'✅ TotalCharges converted. Missing: {df_clean["TotalCharges"].isnull().sum()}')

# 3. Fill missing TotalCharges with median (only 11 rows)
df_clean['TotalCharges'].fillna(df_clean['TotalCharges'].median(), inplace=True)
print(f'✅ Missing TotalCharges filled. Remaining: {df_clean["TotalCharges"].isnull().sum()}')

# 4. Encode target
df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)
print(f'✅ Churn encoded: Yes=1, No=0. Churn rate: {df_clean["Churn"].mean():.1%}')

print(f'\n📊 Clean dataset shape: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')

## 4. Feature Engineering <a id='4'></a>

In [ ]:
# ── 4.1 New Features ──────────────────────────────────────────────────────────
df_feat = df_clean.copy()

# Feature 1: Average monthly spend (TotalCharges / tenure)
df_feat['AvgMonthlySpend'] = np.where(
    df_feat['tenure'] > 0,
    df_feat['TotalCharges'] / df_feat['tenure'],
    df_feat['MonthlyCharges']
)
print('✅ Created: AvgMonthlySpend')

# Feature 2: Tenure groups (binning)
df_feat['TenureGroup'] = pd.cut(df_feat['tenure'],
                                 bins=[0, 12, 24, 48, 72],
                                 labels=['New(0-1yr)', 'Growing(1-2yr)', 'Mature(2-4yr)', 'Loyal(4+yr)'],
                                 include_lowest=True)
print('✅ Created: TenureGroup')

# Feature 3: Number of services subscribed
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df_feat['NumServices'] = (df_feat[service_cols] == 'Yes').sum(axis=1)
print('✅ Created: NumServices')

# Feature 4: Is high value customer?
df_feat['IsHighValue'] = (df_feat['MonthlyCharges'] > df_feat['MonthlyCharges'].median()).astype(int)
print('✅ Created: IsHighValue')

# Feature 5: Has any protection service
protection_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
df_feat['HasProtection'] = (df_feat[protection_cols] == 'Yes').any(axis=1).astype(int)
print('✅ Created: HasProtection')

print(f'\n📊 New features added. Shape: {df_feat.shape}')
df_feat[['AvgMonthlySpend', 'TenureGroup', 'NumServices', 'IsHighValue', 'HasProtection']].head()

In [ ]:
# ── 4.2 Encode All Categoricals ───────────────────────────────────────────────
df_encoded = df_feat.copy()

# Get categorical columns
cat_cols = df_encoded.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns to encode: {cat_cols}')

# Label encode
le = LabelEncoder()
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    
print(f'\n✅ All categoricals encoded!')
print(f'Final shape: {df_encoded.shape}')
df_encoded.dtypes.value_counts()

In [ ]:
# ── 4.3 Train-Test Split ──────────────────────────────────────────────────────
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train set: {X_train.shape} | Churn rate: {y_train.mean():.1%}')
print(f'Test set:  {X_test.shape}  | Churn rate: {y_test.mean():.1%}')
print('\n✅ Data split complete (80/20 stratified)!')

## 5. Model Training <a id='5'></a>

In [ ]:
# ── 5.1 Logistic Regression ───────────────────────────────────────────────────
print('🤖 Training Logistic Regression...')
lr = LogisticRegression(random_state=42, max_iter=1000, C=1.0)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('✅ Logistic Regression trained!')
print(f'   Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'   F1-Score : {f1_score(y_test, y_pred_lr):.4f}')
print(f'   ROC-AUC  : {roc_auc_score(y_test, y_prob_lr):.4f}')

In [ ]:
# ── 5.2 Decision Tree ─────────────────────────────────────────────────────────
print('🌳 Training Decision Tree...')
dt = DecisionTreeClassifier(random_state=42, max_depth=6, min_samples_split=20)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print('✅ Decision Tree trained!')
print(f'   Accuracy : {accuracy_score(y_test, y_pred_dt):.4f}')
print(f'   F1-Score : {f1_score(y_test, y_pred_dt):.4f}')
print(f'   ROC-AUC  : {roc_auc_score(y_test, y_prob_dt):.4f}')

In [ ]:
# ── 5.3 Random Forest ─────────────────────────────────────────────────────────
print('🌲 Training Random Forest...')
rf = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10,
                              min_samples_split=10, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('✅ Random Forest trained!')
print(f'   Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'   F1-Score : {f1_score(y_test, y_pred_rf):.4f}')
print(f'   ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}')

## 6. Model Comparison & Evaluation <a id='6'></a>

In [ ]:
# ── 6.1 Metrics Comparison Table ─────────────────────────────────────────────
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Decision Tree':       (y_pred_dt, y_prob_dt),
    'Random Forest':       (y_pred_rf, y_prob_rf)
}

results = []
for name, (y_pred, y_prob) in models.items():
    results.append({
        'Model': name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4)
    })

results_df = pd.DataFrame(results).set_index('Model')
print('📊 MODEL COMPARISON TABLE')
print('='*60)
print(results_df.to_string())
print('\n🏆 Best Model by F1-Score:', results_df['F1-Score'].idxmax())
print('🏆 Best Model by ROC-AUC: ', results_df['ROC-AUC'].idxmax())

In [ ]:
# ── 6.2 Visual Model Comparison ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Grouped bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics))
width = 0.25
colors_m = ['#3498db', '#e67e22', '#2ecc71']

for i, (model, color) in enumerate(zip(results_df.index, colors_m)):
    vals = [results_df.loc[model, m] for m in metrics]
    bars = axes[0].bar(x + i*width, vals, width, label=model, color=color,
                       alpha=0.85, edgecolor='white')

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Comparison — All Metrics', fontweight='bold', fontsize=13)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0.5, 1.0)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# ROC Curves
model_preds = [
    ('Logistic Regression', y_prob_lr, '#3498db'),
    ('Decision Tree', y_prob_dt, '#e67e22'),
    ('Random Forest', y_prob_rf, '#2ecc71')
]
for name, y_prob, color in model_preds:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparison', fontweight='bold', fontsize=13)
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/06_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.3 Confusion Matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (name, (y_pred, _)), color in zip(axes, models.items(), ['Blues', 'Oranges', 'Greens']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'],
                linewidths=1, linecolor='white', annot_kws={'size': 14})
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\nAccuracy: {acc:.3f}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/07_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6.4 Detailed Classification Reports ──────────────────────────────────────
for name, (y_pred, _) in models.items():
    print(f'\n{'='*55}')
    print(f'  📋 {name}')
    print(f'{'='*55}')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# ── 6.5 Feature Importance (Random Forest) ────────────────────────────────────
feat_importance = pd.Series(rf.feature_importances_, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 7))
colors_fi = sns.color_palette('RdYlGn_r', len(feat_importance))
bars = plt.barh(feat_importance.index[::-1], feat_importance.values[::-1],
                color=colors_fi[::-1], edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, feat_importance.values[::-1]):
    plt.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)

plt.xlabel('Feature Importance', fontsize=12)
plt.title('Top 15 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔑 Top 5 most important features:')
for feat, imp in feat_importance.head(5).items():
    print(f'   {feat:<25} {imp:.4f}')

In [ ]:
# ── 6.6 Cross-Validation (5-fold) ─────────────────────────────────────────────
print('🔁 Running 5-Fold Cross-Validation...\n')

cv_models = [
    ('Logistic Regression', LogisticRegression(random_state=42, max_iter=1000), X_train_scaled),
    ('Decision Tree',       DecisionTreeClassifier(random_state=42, max_depth=6), X_train),
    ('Random Forest',       RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=-1), X_train)
]

cv_results = {}
for name, model, X_data in cv_models:
    scores = cross_val_score(model, X_data, y_train, cv=5, scoring='f1')
    cv_results[name] = scores
    print(f'{name}:')
    print(f'   F1 per fold: {[f"{s:.3f}" for s in scores]}')
    print(f'   Mean ± Std : {scores.mean():.3f} ± {scores.std():.3f}\n')

## 7. Save Best Model <a id='7'></a>

In [ ]:
# ── 7.1 Save Best Model & Artifacts ──────────────────────────────────────────
best_model_name = results_df['ROC-AUC'].idxmax()
print(f'🏆 Best Model: {best_model_name}')

model_map = {
    'Logistic Regression': lr,
    'Decision Tree': dt,
    'Random Forest': rf
}
best_model = model_map[best_model_name]

# Save model, scaler, and feature names
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(list(X.columns), '../models/feature_names.pkl')
results_df.to_csv('../models/model_results.csv')

print('\n✅ Saved:')
print('   ../models/best_model.pkl')
print('   ../models/scaler.pkl')
print('   ../models/feature_names.pkl')
print('   ../models/model_results.csv')

print('\n📊 FINAL RESULTS SUMMARY')
print('='*60)
print(results_df.to_string())

In [ ]:
# ── 7.2 Final Summary ─────────────────────────────────────────────────────────
print('\n' + '='*60)
print('  🎯 PROJECT COMPLETE: Customer Churn Prediction')
print('='*60)
print(f'  Dataset     : IBM Telco Customer Churn ({len(df):,} rows)')
print(f'  Kaggle URL  : https://www.kaggle.com/datasets/blastchar/telco-customer-churn')
print(f'  Churn Rate  : {df["Churn"].apply(lambda x: 1 if x=="Yes" or x==1 else 0).mean():.1%}')
print(f'  Features    : {X.shape[1]} (including 5 engineered)')
print(f'  Best Model  : {best_model_name}')
print(f'  Best AUC    : {results_df.loc[best_model_name, "ROC-AUC"]:.4f}')
print(f'  Best F1     : {results_df.loc[best_model_name, "F1-Score"]:.4f}')
print('='*60)